In [1]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# 01 - Exploration des Données BRCA_TCGA et PANCAN\n",
        "\n",
        "Ce notebook explore les données cliniques et génomiques pour la prédiction d'issues cliniques avec XAI.\n",
        "\n",
        "## Objectifs\n",
        "1. Charger les données cliniques et génomiques.\n",
        "2. Explorer les distributions et relations entre variables.\n",
        "3. Identifier les problèmes potentiels (valeurs manquantes, déséquilibres).\n",
        "4. Générer des visualisations pour le rapport."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# --- Imports ---\n",
        "import yaml\n",
        "import pandas as pd\n",
        "import numpy as np\n",
        "import os\n",
        "import matplotlib.pyplot as plt\n",
        "import seaborn as sns\n",
        "from IPython.display import display\n",
        "from pathlib import Path\n",
        "\n",
        "# Configuration des visualisations\n",
        "%matplotlib inline\n",
        "plt.style.use('seaborn-v0_8')\n",
        "sns.set_palette(\"husl\")\n",
        "sns.set_context(\"notebook\", font_scale=1.1)\n",
        "\n",
        "# Charger la configuration\n",
        "with open(\"../config/config.yaml\", \"r\") as f:\n",
        "    config = yaml.safe_load(f)\n",
        "\n",
        "# Chemins vers les données\n",
        "data_dir = config[\"data\"][\"data_dir\"]\n",
        "pancan_dir = config[\"data\"].get(\"pancan_dir\")\n",
        "target_column = config[\"data\"][\"target_column\"]"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 1. Chargement des Données Cliniques"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Charger les données cliniques BRCA\n",
        "clinical_path = os.path.join(data_dir, config[\"data\"][\"clinical_file\"])\n",
        "df_clinical = pd.read_csv(clinical_path, sep=\"\\t\")\n",
        "\n",
        "print(f\"Données cliniques chargées: {df_clinical.shape}\")\n",
        "display(df_clinical.head())\n",
        "display(df_clinical[target_column].value_counts(dropna=False))"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### 1.1. Statistiques Descriptives"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Statistiques pour les variables numériques\n",
        "numeric_cols = df_clinical.select_dtypes(include=[np.number]).columns\n",
        "print(\"Statistiques des variables numériques:\")\n",
        "display(df_clinical[numeric_cols].describe())\n",
        "\n",
        "# Variables catégorielles\n",
        "categorical_cols = df_clinical.select_dtypes(include=['object', 'category']).columns\n",
        "if len(categorical_cols) > 0:\n",
        "    print(\"\\nDistribution des variables catégorielles:\")\n",
        "    for col in categorical_cols:\n",
        "        print(f\"\\n{col}:\")\n",
        "        display(df_clinical[col].value_counts(dropna=False))"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### 1.2. Visualisation de la Variable Cible"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Distribution de la cible\n",
        "plt.figure(figsize=(12, 5))\n",
        "\n",
        "plt.subplot(1, 2, 1)\n",
        "sns.countplot(x=target_column, data=df_clinical)\n",
        "plt.title(f\"Distribution de {target_column}\")\n",
        "\n",
        "plt.subplot(1, 2, 2)\n",
        "df_clinical[target_column].value_counts().plot(kind='pie', autopct='%1.1f%%')\n",
        "plt.title(\"Proportions\")\n",
        "\n",
        "plt.tight_layout()\n",
        "plt.savefig('../reports/figures/target_distribution.png', dpi=300, bbox_inches='tight')\n",
        "plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### 1.3. Matrice de Corrélation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Matrice de corrélation pour les variables numériques\n",
        "if len(numeric_cols) > 1:\n",
        "    plt.figure(figsize=(12, 10))\n",
        "    corr_matrix = df_clinical[numeric_cols].corr()\n",
        "    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt=\".2f\")\n",
        "    plt.title(\"Matrice de Corrélation des Variables Cliniques\")\n",
        "    plt.savefig('../reports/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')\n",
        "    plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 2. Exploration des Données Génomiques"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Charger les données d'expression BRCA\n",
        "expression_path = os.path.join(data_dir, config[\"data\"][\"genomic_files\"][2])  # data_mrna_seq_v2_rsem.tsv\n",
        "df_expression = pd.read_csv(expression_path, sep=\"\\t\", index_col=0, nrows=100)  # Lire 100 lignes pour l'exemple\n",
        "\n",
        "print(f\"\\nDonnées d'expression BRCA: {df_expression.shape}\")\n",
        "display(df_expression.head())"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### 2.1. Distribution de l'Expression de Gènes Clés"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Liste de gènes d'intérêt (exemples)\n",
        "genes_of_interest = [\"BRCA1\", \"BRCA2\", \"TP53\", \"ERBB2\", \"PIK3CA\"]\n",
        "\n",
        "# Visualiser la distribution pour chaque gène\n",
        "for gene in genes_of_interest:\n",
        "    if gene in df_expression.index:\n",
        "        plt.figure(figsize=(8, 4))\n",
        "        sns.boxplot(data=df_expression.loc[gene].to_frame().T)\n",
        "        plt.title(f\"Distribution de l'expression de {gene}\")\n",
        "        plt.savefig(f'../reports/figures/expression_{gene}.png', dpi=300, bbox_inches='tight')\n",
        "        plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### 2.2. Chargement des Données PANCAN (si disponibles)"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "if pancan_dir:\n",
        "    # Charger les données d'expression PANCAN\n",
        "    pancan_expression_path = os.path.join(pancan_dir, config[\"data\"][\"pancan_files\"][\"expression\"])\n",
        "    df_pancan = pd.read_csv(pancan_expression_path, sep=\"\\t\", index_col=0, nrows=100)\n",
        "    print(f\"\\nDonnées PANCAN (expression): {df_pancan.shape}\")\n",
        "    \n",
        "    # Filtrer pour BRCA\n",
        "    brca_samples = [col for col in df_pancan.columns if \"BRCA\" in col]\n",
        "    df_pancan_brca = df_pancan[brca_samples]\n",
        "    print(f\"Échantillons BRCA dans PANCAN: {len(brca_samples)}\")\n",
        "    \n",
        "    # Comparer un gène entre BRCA_TCGA et PANCAN\n",
        "    if \"BRCA1\" in df_pancan_brca.index and \"BRCA1\" in df_expression.index:\n",
        "        plt.figure(figsize=(10, 6))\n",
        "        sns.boxplot(data=pd.concat([\n",
        "            df_expression.loc[\"BRCA1\"].to_frame().T.assign(source='BRCA_TCGA'),\n",
        "            df_pancan_brca.loc[\"BRCA1\"].to_frame().T.assign(source='PANCAN')\n",
        "        ], axis=0))\n",
        "        plt.title(\"Comparaison de l'expression de BRCA1 entre BRCA_TCGA et PANCAN\")\n",
        "        plt.savefig('../reports/figures/compare_brca1_expression.png', dpi=300, bbox_inches='tight')\n",
        "        plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 3. Analyse des Valeurs Manquantes"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Valeurs manquantes dans les données cliniques\n",
        "missing_clinical = df_clinical.isna().sum()\n",
        "missing_clinical = missing_clinical[missing_clinical > 0].sort_values(ascending=False)\n",
        "print(\"Valeurs manquantes dans les données cliniques:\")\n",
        "display(missing_clinical)\n",
        "\n",
        "# Visualisation\n",
        "if len(missing_clinical) > 0:\n",
        "    plt.figure(figsize=(10, 6))\n",
        "    sns.barplot(x=missing_clinical.values, y=missing_clinical.index)\n",
        "    plt.title(\"Valeurs manquantes par variable clinique\")\n",
        "    plt.savefig('../reports/figures/missing_values_clinical.png', dpi=300, bbox_inches='tight')\n",
        "    plt.show()\n",
        "\n",
        "# Valeurs manquantes dans les données d'expression\n",
        "missing_expression = df_expression.isna().sum().sum()\n",
        "print(f\"\\nValeurs manquantes dans l'expression: {missing_expression}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 4. Sauvegarde des Résultats de l'Exploration"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Sauvegarder un résumé des données\n",
        "os.makedirs(\"../reports/exploration\", exist_ok=True)\n",
        "\n",
        "with open(\"../reports/exploration/summary.txt\", \"w\") as f:\n",
        "    f.write(\"=== RÉSUMÉ DE L'EXPLORATION ===\\n\\n\")\n",
        "    f.write(f\"Données cliniques: {df_clinical.shape}\\n\")\n",
        "    f.write(f\"Valeurs manquantes cliniques: {df_clinical.isna().sum().sum()}\\n\")\n",
        "    f.write(f\"Données d'expression: {df_expression.shape}\\n\")\n",
        "    f.write(f\"Valeurs manquantes expression: {df_expression.isna().sum().sum()}\\n\")\n",
        "    if pancan_dir:\n",
        "        f.write(f\"Données PANCAN: {df_pancan.shape}\\n\")\n",
        "        f.write(f\"Échantillons BRCA dans PANCAN: {len(brca_samples)}\\n\")\n",
        "\n",
        "# Sauvegarder un échantillon des données pour référence\n",
        "df_clinical.sample(100).to_csv(\"../reports/exploration/clinical_sample.csv\", index=False)\n",
        "df_expression.sample(100, axis=1).to_csv(\"../reports/exploration/expression_sample.csv\")\n",
        "\n",
        "print(\"✅ Exploration terminée. Résultats sauvegardés dans reports/exploration/\")"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.9.0"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 4
}

{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['# 01 - Exploration des Données BRCA_TCGA et PANCAN\n',
    '\n',
    "Ce notebook explore les données cliniques et génomiques pour la prédiction d'issues cliniques avec XAI.\n",
    '\n',
    '## Objectifs\n',
    '1. Charger les données cliniques et génomiques.\n',
    '2. Explorer les distributions et relations entre variables.\n',
    '3. Identifier les problèmes potentiels (valeurs manquantes, déséquilibres).\n',
    '4. Générer des visualisations pour le rapport.']},
  {'cell_type': 'code',
   'execution_count': None,
   'metadata': {},
   'outputs': [],
   'source': ['# --- Imports ---\n',
    'import yaml\n',
    'import pandas as pd\n',
    'import numpy as np\n',
    'import os\n',
    'import matplotlib.pyplot as plt\n',
    'import seaborn as sns\n',
    'from IPython.display import display\n',
    'from pathlib import Path\n',
    '\n',
    '# Configuration des visualisations\n',
    '%matplotlib inline\